# HyperPLQY Explorer - Notebook

Interactive exploration of spatially-resolved, power-dependent PLQY data.

Download `PLQY_absolute_vs_suns.h5` from the [GitHub Releases](https://github.com/dubajicmilos/hyper-plqy-explorer/releases/latest) and place it in the same directory as this notebook.

In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt

H5_FILE = 'PLQY_absolute_vs_suns.h5'

with h5py.File(H5_FILE, 'r') as f:
    plqy = f['PLQY_percent'][:]       # (20, 1006, 1006)
    suns = f['suns'][:]               # (20,)
    sat_valid = f['saturation_valid'][:] if 'saturation_valid' in f else None

n_suns, H, W = plqy.shape
print(f'PLQY cube: {plqy.shape}')
print(f'Excitation intensities: {suns} suns')

## 1. PLQY maps at selected excitation intensities

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(22, 4))

indices = [1, 5, 9, 14, 19]  # skip index 0 (1 sun, near lasing threshold)

for ax, idx in zip(axes, indices):
    img = plqy[idx]
    vmin, vmax = np.percentile(img, [3, 97])
    im = ax.imshow(img, origin='lower', cmap='inferno', vmin=vmin, vmax=vmax)
    ax.set_title(f'{suns[idx]:.1f} suns')
    ax.axis('off')
    plt.colorbar(im, ax=ax, shrink=0.8, label='PLQY %')

fig.suptitle('Absolute PLQY maps at selected excitation intensities', fontsize=14)
plt.tight_layout()
plt.show()

## 2. PLQY vs excitation intensity for custom ROIs

Define rectangular ROIs below. Each is `(y_start, y_end, x_start, x_end)` in pixel coordinates.

In [ ]:
# ── Define your ROIs here ──────────────────────────────────────
# Format: 'name': (y_start, y_end, x_start, x_end)
rois = {
    'Bright region':  (550, 650, 350, 450),
    'Dark region':    (300, 400, 550, 650),
    'Edge':           (200, 300, 250, 350),
}

colors = ['#e6194b', '#3cb44b', '#4363d8', '#f58231', '#911eb4']

# ── Compute mean PLQY vs suns for each ROI ────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Left: show ROIs on the map
ref_idx = np.argmin(np.abs(suns - 27.0))
ref_img = plqy[ref_idx]
vmin, vmax = np.percentile(ref_img, [3, 97])
ax1.imshow(ref_img, origin='lower', cmap='inferno', vmin=vmin, vmax=vmax)
ax1.set_title(f'ROIs on PLQY map ({suns[ref_idx]:.0f} suns)')
ax1.axis('off')

for i, (name, (y0, y1, x0, x1)) in enumerate(rois.items()):
    color = colors[i % len(colors)]
    rect = plt.Rectangle((x0, y0), x1 - x0, y1 - y0,
                          linewidth=2, edgecolor=color, facecolor='none')
    ax1.add_patch(rect)
    ax1.text(x0 + 2, y1 + 8, name, color=color, fontsize=9, fontweight='bold')

    # Right: PLQY vs suns curve
    mean_plqy = np.array([np.mean(plqy[j, y0:y1, x0:x1]) for j in range(n_suns)])
    n_px = (y1 - y0) * (x1 - x0)
    # Plot 6.2+ suns as filled circles, 1 sun as open circle
    ax2.plot(suns[1:], mean_plqy[1:], 'o-', color=color,
             label=f'{name} ({n_px} px)', markersize=5, linewidth=1.5)
    ax2.plot(suns[0], mean_plqy[0], 'o', color=color,
             markersize=7, markerfacecolor='none', markeredgewidth=1.5)

ax2.set_xlabel('Excitation intensity (suns)')
ax2.set_ylabel('PLQY (%)')
ax2.set_title('Mean PLQY vs excitation intensity')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. PLQY histogram at a chosen intensity

In [ ]:
sun_value = 27.0  # change this to explore different intensities
idx = np.argmin(np.abs(suns - sun_value))

img = plqy[idx]
# Use pixels with PLQY > 0 (on-crystal)
vals = img[img > 0.01]

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(vals, bins=100, color='steelblue', edgecolor='none')
ax.axvline(np.median(vals), color='red', ls='--',
           label=f'median = {np.median(vals):.2f}%')
ax.set_xlabel('PLQY (%)')
ax.set_ylabel('Pixel count')
ax.set_title(f'PLQY distribution at {suns[idx]:.1f} suns')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Single-pixel spectrum

Pick a pixel and see how its PLQY varies with excitation power.

In [ ]:
# Choose pixel coordinates
py, px = 500, 400

pixel_plqy = plqy[:, py, px]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(suns[1:], pixel_plqy[1:], 'o-', color='steelblue', markersize=6)
ax.plot(suns[0], pixel_plqy[0], 'o', color='steelblue',
        markersize=8, markerfacecolor='none', markeredgewidth=1.5)
ax.set_xlabel('Excitation intensity (suns)')
ax.set_ylabel('PLQY (%)')
ax.set_title(f'PLQY vs suns at pixel ({py}, {px})')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()